In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Optional

import pandas as pd
import matplotlib.pyplot as plt


@dataclass(frozen=True)
class Round1Paths:
    root: Path = Path("ROUND1")

    def prices_path(self, day: int) -> Path:
        return self.root / f"prices_round_1_day_{day}.csv"

    def trades_path(self, day: int) -> Path:
        return self.root / f"trades_round_1_day_{day}.csv"


def load_prices(days: Iterable[int], base: Path = Path(".")) -> pd.DataFrame:
    """Load and concat price-book snapshots across requested days."""
    p = Round1Paths()
    frames = []
    for d in days:
        path = base / p.prices_path(d)
        df = pd.read_csv(path, sep=";")
        df["day"] = df["day"].astype(int)
        df["timestamp"] = df["timestamp"].astype(int)
        frames.append(df)
    out = pd.concat(frames, ignore_index=True)
    return out.sort_values(["day", "timestamp", "product"], kind="stable").reset_index(drop=True)


def load_trades(days: Iterable[int], base: Path = Path(".")) -> pd.DataFrame:
    """Load and concat market trades across requested days."""
    p = Round1Paths()
    frames = []
    for d in days:
        path = base / p.trades_path(d)
        df = pd.read_csv(path, sep=";")
        df["timestamp"] = df["timestamp"].astype(int)
        df["day"] = d
        frames.append(df)
    out = pd.concat(frames, ignore_index=True)
    return out.sort_values(["day", "timestamp", "symbol"], kind="stable").reset_index(drop=True)


def _filter_product(df: pd.DataFrame, product: str, product_col: str = "product") -> pd.DataFrame:
    return df[df[product_col] == product].copy()


def plot_midprice(prices: pd.DataFrame, product: str, day: Optional[int] = None, ax=None):
    """Line plot of mid_price vs timestamp."""
    df = _filter_product(prices, product, "product")
    if day is not None:
        df = df[df["day"] == day]
        title = f"{product} mid_price (day {day})"
    else:
        title = f"{product} mid_price (all loaded days)"

    ax = ax or plt.gca()
    for d, g in df.groupby("day"):
        ax.plot(g["timestamp"], g["mid_price"], label=f"day {d}")
    ax.set_title(title)
    ax.set_xlabel("timestamp")
    ax.set_ylabel("mid_price")
    ax.legend()
    return ax


def plot_spread(prices: pd.DataFrame, product: str, day: Optional[int] = None, ax=None):
    """Line plot of (best ask - best bid) vs timestamp."""
    df = _filter_product(prices, product, "product")
    df = df.dropna(subset=["bid_price_1", "ask_price_1"]).copy()
    df["spread"] = df["ask_price_1"] - df["bid_price_1"]
    if day is not None:
        df = df[df["day"] == day]
        title = f"{product} spread (day {day})"
    else:
        title = f"{product} spread (all loaded days)"

    ax = ax or plt.gca()
    for d, g in df.groupby("day"):
        ax.plot(g["timestamp"], g["spread"], label=f"day {d}")
    ax.set_title(title)
    ax.set_xlabel("timestamp")
    ax.set_ylabel("ask_price_1 - bid_price_1")
    ax.legend()
    return ax


def plot_top_of_book(prices: pd.DataFrame, product: str, day: int, n: int = 300):
    """Plot best bid/ask and mid over a short window (first n rows for that day/product)."""
    df = _filter_product(prices, product, "product")
    df = df[df["day"] == day].copy()
    df = df.head(n)

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(df["timestamp"], df["bid_price_1"], label="best_bid", alpha=0.9)
    ax.plot(df["timestamp"], df["ask_price_1"], label="best_ask", alpha=0.9)
    ax.plot(df["timestamp"], df["mid_price"], label="mid", linewidth=2)
    ax.set_title(f"{product} top-of-book (day {day}, first {len(df)} rows)")
    ax.set_xlabel("timestamp")
    ax.set_ylabel("price")
    ax.legend()
    fig.tight_layout()
    return fig, ax


def plot_trade_prints(trades: pd.DataFrame, product: str, day: int, ax=None):
    """Scatter plot of trade price vs timestamp (size = quantity)."""
    df = trades[(trades["day"] == day) & (trades["symbol"] == product)].copy()
    ax = ax or plt.gca()
    ax.scatter(df["timestamp"], df["price"], s=(df["quantity"].abs() * 5).clip(5, 200), alpha=0.5)
    ax.set_title(f"{product} trade prints (day {day})")
    ax.set_xlabel("timestamp")
    ax.set_ylabel("trade price")
    return ax


def plot_returns(prices: pd.DataFrame, product: str, day: int, ax=None):
    """Simple mid-price returns over time."""
    df = _filter_product(prices, product, "product")
    df = df[df["day"] == day].copy()
    df = df.sort_values("timestamp")
    df["ret"] = df["mid_price"].pct_change().fillna(0.0)

    ax = ax or plt.gca()
    ax.plot(df["timestamp"], df["ret"], linewidth=1)
    ax.set_title(f"{product} mid-price returns (day {day})")
    ax.set_xlabel("timestamp")
    ax.set_ylabel("pct_change(mid_price)")
    return ax


plt.style.use("seaborn-v0_8")
pd.set_option("display.max_columns", 50)
print("Loaded helpers: load_prices, load_trades, plot_midprice, plot_spread, plot_top_of_book, plot_trade_prints, plot_returns")


Loaded helpers: load_prices, load_trades, plot_midprice, plot_spread, plot_top_of_book, plot_trade_prints, plot_returns


In [2]:
# Example: load all Round 1 days you have (-2, -1, 0)
prices = load_prices(days=[-2, -1, 0])
trades = load_trades(days=[-2, -1, 0])

prices.head()


FileNotFoundError: [Errno 2] No such file or directory: 'ROUND1\\prices_round_1_day_-2.csv'

In [ ]:
# Example plots (adjust product/day as needed)
fig, ax = plt.subplots(2, 2, figsize=(12, 8))

plot_midprice(prices, product="ASH_COATED_OSMIUM", day=0, ax=ax[0, 0])
plot_spread(prices, product="ASH_COATED_OSMIUM", day=0, ax=ax[0, 1])
plot_trade_prints(trades, product="ASH_COATED_OSMIUM", day=0, ax=ax[1, 0])
plot_returns(prices, product="ASH_COATED_OSMIUM", day=0, ax=ax[1, 1])

fig.tight_layout()
plt.show()
